# AI Programming — Lecture 19
## Further Studies 2: Multivariate Channel-Independent Patch Transformer

ETTh1의 여러 channel을 사용하지만,
각 channel을 **독립적인 sequence**로 처리하고 동일한 Transformer를 공유합니다.

### 핵심 아이디어
```text
Multivariate input
→ channel별 patch 생성
→ 각 channel을 독립 sample처럼 변환
→ shared Patch Transformer
→ channel별 forecast
→ OT만 평가
```

이 방식을 **Channel-Independent (CI)** 처리라고 합니다.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
from keras import layers

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED = 42
keras.utils.set_random_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", gpus)


## 1. ETTh1 데이터 불러오기

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = Path('/content/drive/MyDrive/Colab Notebooks/data/ETTh1.csv')

df = pd.read_csv(DATA_PATH)

CHANNELS = [
    'HUFL', 'HULL', 'MUFL', 'MULL',
    'LUFL', 'LULL', 'OT'
]

values = df[CHANNELS].values.astype('float32')

print("Data path:", DATA_PATH)
print("Shape:", df.shape)
print("Channels:", CHANNELS)
print(df.head())


## 2. Chronological Split과 Standardization

In [ ]:
LOOKBACK = 96
PRED_LEN = 24
NUM_CHANNELS = len(CHANNELS)

n = len(values)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_raw = values[:train_end]
val_raw = values[train_end:val_end]
test_raw = values[val_end:]

scaler = StandardScaler()

train_scaled = scaler.fit_transform(train_raw)
val_scaled = scaler.transform(val_raw)
test_scaled = scaler.transform(test_raw)

print("Train:", train_scaled.shape)
print("Validation:", val_scaled.shape)
print("Test:", test_scaled.shape)


## 3. Multivariate Forecasting Window 생성

In [ ]:
def create_windows(values, lookback=96, pred_len=24):
    X, Y, Y_res = [], [], []
    total_len = lookback + pred_len

    for i in range(len(values) - total_len + 1):
        window = values[i:i + total_len]

        past = window[:lookback]
        future = window[lookback:]

        # Last observed value for each channel
        baseline = past[-1]

        X.append(past)
        Y.append(future)
        Y_res.append(future - baseline)

    return (
        np.array(X, dtype='float32'),
        np.array(Y, dtype='float32'),
        np.array(Y_res, dtype='float32')
    )

X_train, y_train, y_train_res = create_windows(
    train_scaled, LOOKBACK, PRED_LEN
)
X_val, y_val, y_val_res = create_windows(
    val_scaled, LOOKBACK, PRED_LEN
)
X_test, y_test, y_test_res = create_windows(
    test_scaled, LOOKBACK, PRED_LEN
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)


## 4. Channel-Independent Overlapping Patches

In [ ]:
PATCH_LEN = 16
STRIDE = 8

N_PATCHES = (LOOKBACK - PATCH_LEN) // STRIDE + 1

def create_multivariate_patches(X, patch_len=16, stride=8):
    # X: [B, T, C]
    channel_patches = []

    for c in range(X.shape[2]):
        patches = []

        for start in range(0, LOOKBACK - patch_len + 1, stride):
            patch = X[:, start:start + patch_len, c]
            patches.append(patch)

        # [B, N_PATCHES, PATCH_LEN]
        patches = np.stack(patches, axis=1)
        channel_patches.append(patches)

    # [B, C, N_PATCHES, PATCH_LEN]
    return np.stack(channel_patches, axis=1).astype('float32')

X_train_patch = create_multivariate_patches(
    X_train, PATCH_LEN, STRIDE
)
X_val_patch = create_multivariate_patches(
    X_val, PATCH_LEN, STRIDE
)
X_test_patch = create_multivariate_patches(
    X_test, PATCH_LEN, STRIDE
)

print("Number of patches per channel:", N_PATCHES)
print("Patched train shape:", X_train_patch.shape)


## 5. Channel을 Independent Sequence로 변환

원래 shape의 channel 축을 batch 축과 합쳐
동일한 shared Transformer가 모든 channel을 처리하도록 합니다.

In [ ]:
def to_channel_independent_inputs(X_patch, y_res):
    B = X_patch.shape[0]

    # [B, C, N, P] -> [B*C, N, P]
    X_ci = X_patch.reshape(
        B * NUM_CHANNELS,
        N_PATCHES,
        PATCH_LEN
    )

    # [B, H, C] -> [B, C, H] -> [B*C, H]
    y_ci = np.transpose(
        y_res,
        (0, 2, 1)
    ).reshape(
        B * NUM_CHANNELS,
        PRED_LEN
    )

    return X_ci.astype('float32'), y_ci.astype('float32')

X_train_ci, y_train_ci = to_channel_independent_inputs(
    X_train_patch, y_train_res
)
X_val_ci, y_val_ci = to_channel_independent_inputs(
    X_val_patch, y_val_res
)
X_test_ci, y_test_ci = to_channel_independent_inputs(
    X_test_patch, y_test_res
)

print("Channel-independent train X:", X_train_ci.shape)
print("Channel-independent train y:", y_train_ci.shape)


## 6. Learned Positional Embedding

In [ ]:
class LearnedPositionalEmbedding(layers.Layer):
    def __init__(self, max_len, embed_dim):
        super().__init__()

        self.position_embedding = layers.Embedding(
            input_dim=max_len,
            output_dim=embed_dim
        )

    def call(self, inputs):
        positions = keras.ops.arange(
            0, keras.ops.shape(inputs)[1], 1
        )

        return inputs + self.position_embedding(positions)


## 7. Transformer Encoder Block

In [ ]:
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()

        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout
        )

        self.dense1 = layers.Dense(
            ff_dim,
            activation='relu'
        )
        self.dense2 = layers.Dense(embed_dim)

        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()

        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)

    def call(self, inputs, training=None):
        attention_output = self.attention(
            inputs,
            inputs,
            training=training
        )

        x = self.norm1(
            inputs
            + self.dropout1(
                attention_output,
                training=training
            )
        )

        ffn_output = self.dense2(
            self.dense1(x)
        )

        return self.norm2(
            x
            + self.dropout2(
                ffn_output,
                training=training
            )
        )


## 8. Shared Patch Transformer 구성

In [ ]:
EMBED_DIM = 64
NUM_HEADS = 4
FF_DIM = 128
DROPOUT = 0.1

inputs = keras.Input(
    shape=(N_PATCHES, PATCH_LEN)
)

projection_layer = layers.Dense(EMBED_DIM)
x = projection_layer(inputs)

position_layer = LearnedPositionalEmbedding(
    N_PATCHES, EMBED_DIM
)
x = position_layer(x)

encoder1 = TransformerEncoder(
    EMBED_DIM, NUM_HEADS, FF_DIM, DROPOUT
)
x = encoder1(x)

encoder2 = TransformerEncoder(
    EMBED_DIM, NUM_HEADS, FF_DIM, DROPOUT
)
x = encoder2(x)

flatten_layer = layers.Flatten()
x = flatten_layer(x)

output_layer = layers.Dense(PRED_LEN)
outputs = output_layer(x)

model = keras.Model(
    inputs,
    outputs,
    name='channel_independent_patch_transformer'
)

model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss='mse',
    metrics=['mae']
)

model.summary()


## 9. Model Training

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train_ci,
    y_train_ci,
    validation_data=(
        X_val_ci,
        y_val_ci
    ),
    epochs=100,
    batch_size=128,
    shuffle=False,
    callbacks=[early_stopping],
    verbose=1
)


In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(
    history.history['loss'],
    label='Train'
)
plt.plot(
    history.history['val_loss'],
    label='Validation'
)

plt.xlabel('Epoch')
plt.ylabel('Residual MSE')
plt.title('Learning Curve')
plt.legend()
plt.grid(True)
plt.show()


## 10. OT-Only Evaluation

모든 channel로 shared model을 학습하지만,
최종 평가는 target channel인 **OT**에 대해서만 수행합니다.

In [ ]:
pred_res_ci = model.predict(
    X_test_ci,
    batch_size=512,
    verbose=1
)

B_test = X_test.shape[0]

# [B*C, H] -> [B, C, H] -> [B, H, C]
pred_res = pred_res_ci.reshape(
    B_test,
    NUM_CHANNELS,
    PRED_LEN
)
pred_res = np.transpose(
    pred_res,
    (0, 2, 1)
)

# Reconstruct absolute predictions for all channels.
baseline = X_test[:, -1, :][:, None, :]
y_pred = baseline + pred_res

# --------------------------------------------------
# Evaluate OT only
# --------------------------------------------------
OT_IDX = CHANNELS.index('OT')

y_test_ot = y_test[:, :, OT_IDX]
y_pred_ot = y_pred[:, :, OT_IDX]

ot_norm_mse = mean_squared_error(
    y_test_ot.reshape(-1),
    y_pred_ot.reshape(-1)
)
ot_norm_mae = mean_absolute_error(
    y_test_ot.reshape(-1),
    y_pred_ot.reshape(-1)
)

# Convert OT back to the original scale.
ot_mean = scaler.mean_[OT_IDX]
ot_scale = scaler.scale_[OT_IDX]

y_test_ot_real = (
    y_test_ot * ot_scale + ot_mean
)
y_pred_ot_real = (
    y_pred_ot * ot_scale + ot_mean
)

ot_real_mse = mean_squared_error(
    y_test_ot_real.reshape(-1),
    y_pred_ot_real.reshape(-1)
)
ot_real_mae = mean_absolute_error(
    y_test_ot_real.reshape(-1),
    y_pred_ot_real.reshape(-1)
)

print('OT-Only Performance')
print(f'Normalized MSE : {ot_norm_mse:.4f}')
print(f'Normalized MAE : {ot_norm_mae:.4f}')
print(f'MSE (°C²)      : {ot_real_mse:.4f}')
print(f'MAE (°C)       : {ot_real_mae:.4f}')


## 11. Further Studies 1과 비교

In [ ]:
fs1_norm_mse = 0.0507
fs1_norm_mae = 0.1654
fs1_real_mse = 3.5340
fs1_real_mae = 1.3808

print('Further Studies 1: Univariate OT')
print(f'Normalized MSE : {fs1_norm_mse:.4f}')
print(f'Normalized MAE : {fs1_norm_mae:.4f}')
print(f'MSE (°C²)      : {fs1_real_mse:.4f}')
print(f'MAE (°C)       : {fs1_real_mae:.4f}')

print()
print('Further Studies 2: Multivariate Shared Training / OT Evaluation')
print(f'Normalized MSE : {ot_norm_mse:.4f}')
print(f'Normalized MAE : {ot_norm_mae:.4f}')
print(f'MSE (°C²)      : {ot_real_mse:.4f}')
print(f'MAE (°C)       : {ot_real_mae:.4f}')


## 12. OT Forecast Example

In [ ]:
sample_idx = 0

past_ot_real = (
    X_test[sample_idx, :, OT_IDX] * ot_scale
    + ot_mean
)

future_ot_real = y_test_ot_real[sample_idx]
pred_ot_real = y_pred_ot_real[sample_idx]

past_x = np.arange(-LOOKBACK + 1, 1)
future_x = np.arange(1, PRED_LEN + 1)

plt.figure(figsize=(10, 4))

plt.plot(
    past_x,
    past_ot_real,
    label='Past OT'
)
plt.plot(
    future_x,
    future_ot_real,
    label='Ground Truth'
)
plt.plot(
    future_x,
    pred_ot_real,
    label='Patch Transformer'
)

plt.axvline(0, linestyle='--')
plt.xlabel('Time Step')
plt.ylabel('OT (°C)')
plt.title('OT Forecast: Multivariate Shared Training')
plt.legend()
plt.grid(True)
plt.show()


## 13. Channel-Independent란?

Channel 간 attention을 직접 수행하지 않습니다.

대신 모든 channel이 **같은 Transformer parameter를 공유**하면서
각 channel의 temporal pattern을 독립적으로 학습합니다.

## 14. 다음 단계: RevIN

다음 Further Studies에서는
각 sample/channel window를 개별적으로 normalization하는 RevIN을 추가합니다.

## 핵심 정리

- Multivariate data를 사용합니다.
- Channel 간 직접적인 interaction은 사용하지 않습니다.
- Shared Transformer가 각 channel을 동일한 방식으로 처리합니다.
- 최종 성능은 OT channel에서 평가합니다.